
# 05 | Player Performance Analysis

**Objective:** I analyze individual player performance during the 2026 FIFA World Cup, focusing on appearances, goals, playing time, and contributions to team performance.

I use SQL to connect player, match, and appearance datasets, identify top performers, and compare players across national teams.



## 5.1 | Connecting to the Database

I connect to my existing SQLite database to explore player-level statistics and prepare the data for SQL analysis.


In [1]:

import pandas as pd
import sqlite3
from pathlib import Path

# Connect to existing SQLite database
db_path = Path.cwd().parent / "Data" / "Prepared" / "world_cup_2026.db"

if not db_path.is_file():
    raise FileNotFoundError(f"Database not found: {db_path}")

conn = sqlite3.connect(db_path)

print("Connected to World Cup database!")


Connected to World Cup database!



## 5.2 | Exploring Player Datasets

I review the player and appearance tables to identify available statistics and understand how individual players are connected to teams and matches.


In [2]:

# Inspect player-related tables
tables = ["players", "match_appearances", "player_events"]

for table in tables:
    print(f"\nTABLE: {table}")

    query = f'PRAGMA table_info("{table}");'
    display(pd.read_sql_query(query, conn)[["name", "type"]])

    # Preview sample records
    query = f'SELECT * FROM "{table}" LIMIT 3;'
    display(pd.read_sql_query(query, conn))



TABLE: players


,name,type
0,appearances_count,INTEGER
1,display_name,TEXT
2,player_id,TEXT
3,shirt_numbers,INTEGER
4,team_id,TEXT


,appearances_count,display_name,player_id,shirt_numbers,team_id
0,4,Achref ABADA,ALG_achref_abada,3,ALG
1,4,Adil BOULBINA,ALG_adil_boulbina,20,ALG
2,4,Aissa MANDI,ALG_aissa_mandi,2,ALG



TABLE: match_appearances


,name,type
0,appearance_id,TEXT
1,appeared,INTEGER
2,goal_minutes,TEXT
3,is_starter,INTEGER
4,match_id,TEXT
5,match_team_id,TEXT
6,own_goal_minutes,REAL
7,player_id,TEXT
8,player_name,TEXT
9,position,TEXT


,appearance_id,appeared,goal_minutes,is_starter,match_id,match_team_id,own_goal_minutes,player_id,player_name,position,red_card_minutes,shirt_number,status,subbed_off_minutes,subbed_on_minutes,team,team_id,team_role,yellow_card_minutes
0,2026-M001-MEX-RSA_MEX_01,1,None,1,2026-M001-MEX-RSA,2026-M001-MEX-RSA_MEX,None,MEX_raul_rangel,Raul RANGEL,GK,NaN,1,starting,None,None,Mexico,MEX,home,None
1,2026-M001-MEX-RSA_MEX_03,1,None,1,2026-M001-MEX-RSA,2026-M001-MEX-RSA_MEX,None,MEX_cesar_montes,Cesar MONTES,DF,92.0,3,starting,None,None,Mexico,MEX,home,None
2,2026-M001-MEX-RSA_MEX_05,1,None,1,2026-M001-MEX-RSA,2026-M001-MEX-RSA_MEX,None,MEX_johan_vasquez,Johan VASQUEZ,DF,NaN,5,starting,None,None,Mexico,MEX,home,None



TABLE: player_events


,name,type
0,appearance_id,TEXT
1,event_id,TEXT
2,event_type,TEXT
3,match_id,TEXT
4,match_team_id,TEXT
5,minute,INTEGER
6,minute_raw,TEXT
7,player_id,TEXT
8,player_name,TEXT
9,shirt_number,INTEGER


,appearance_id,event_id,event_type,match_id,match_team_id,minute,minute_raw,player_id,player_name,shirt_number,source_page,team,team_id
0,2026-M001-MEX-RSA_MEX_16,2026-M001-MEX-RSA_3ecf3646c282,goal,2026-M001-MEX-RSA,2026-M001-MEX-RSA_MEX,9,9',MEX_julian_quinones,Julian QUINONES,16,2,Mexico,MEX
1,2026-M001-MEX-RSA_RSA_04,2026-M001-MEX-RSA_b6d404c3ffbd,yellow_card,2026-M001-MEX-RSA,2026-M001-MEX-RSA_RSA,17,17',RSA_teboho_mokoena,Teboho MOKOENA,4,2,South Africa,RSA
2,2026-M001-MEX-RSA_MEX_26,2026-M001-MEX-RSA_046c0206230a,yellow_card,2026-M001-MEX-RSA,2026-M001-MEX-RSA_MEX,23,23',MEX_brian_gutierrez,Brian GUTIERREZ,26,2,Mexico,MEX



## 5.3 | Joining Player and Appearance Data

I connect player records with match appearances and national teams to calculate how many matches each player participated in during the tournament.


In [3]:

# Calculate player appearances by national team
query = """
SELECT
    p.display_name AS player,
    t.team_name AS team,
    COUNT(ma.appearance_id) AS matches_played,
    SUM(CASE
        WHEN ma.is_starter = 1 THEN 1
        ELSE 0
    END) AS matches_started

FROM players p

JOIN match_appearances ma
    ON p.player_id = ma.player_id

JOIN teams t
    ON p.team_id = t.team_id

WHERE ma.appeared = 1

GROUP BY
    p.player_id,
    p.display_name,
    t.team_name

ORDER BY matches_played DESC, matches_started DESC
LIMIT 20;
"""

appearances_df = pd.read_sql_query(query, conn)

display(appearances_df)


,player,team,matches_played,matches_started
0,Emiliano MARTINEZ,Argentina,8,8
1,Aymeric LAPORTE,Spain,8,8
2,Marc CUCURELLA,Spain,8,8
3,Mikel OYARZABAL,Spain,8,8
4,Pau CUBARSI,Spain,8,8
5,RODRI,Spain,8,8
6,Unai SIMON,Spain,8,8
7,Kylian MBAPPE,France,8,8
8,Michael OLISE,France,8,8
9,Mike MAIGNAN,France,8,8



## 5.4 | Identifying Top Goal Scorers

I analyze recorded goal events to identify the leading scorers and compare individual attacking contributions across national teams.


In [4]:

# Identify leading goal scorers
query = """
SELECT
    p.display_name AS player,
    t.team_name AS team,
    COUNT(pe.event_id) AS goals_scored

FROM player_events pe

JOIN players p
    ON pe.player_id = p.player_id

JOIN teams t
    ON p.team_id = t.team_id

WHERE pe.event_type = 'goal'

GROUP BY
    p.player_id,
    p.display_name,
    t.team_name

ORDER BY goals_scored DESC, player ASC
LIMIT 20;
"""

scorers_df = pd.read_sql_query(query, conn)

display(scorers_df)


,player,team,goals_scored
0,Kylian MBAPPE,France,10
1,Lionel MESSI,Argentina,8
2,Erling HAALAND,Norway,7
3,Jude BELLINGHAM,England,7
4,Harry KANE,England,6
5,Mikel OYARZABAL,Spain,5
6,Ismaila SARR,Senegal,4
7,Julian QUINONES,Mexico,4
8,VINICIUS JUNIOR,Brazil,4
9,Bradley BARCOLA,France,3



## 5.5 | Comparing Player Scoring Efficiency

I compare individual scoring totals with match appearances to evaluate which players scored most efficiently during the tournament.


In [5]:

# Calculate goals per match for individual players
query = """
WITH appearances AS (
    SELECT
        player_id,
        COUNT(DISTINCT match_id) AS matches_played
    FROM match_appearances
    WHERE appeared = 1
    GROUP BY player_id
),

goals AS (
    SELECT
        player_id,
        COUNT(*) AS goals_scored
    FROM player_events
    WHERE event_type = 'goal'
    GROUP BY player_id
)

SELECT
    p.display_name AS player,
    t.team_name AS team,
    a.matches_played,
    COALESCE(g.goals_scored, 0) AS goals_scored,

    ROUND(
        1.0 * COALESCE(g.goals_scored, 0)
        / NULLIF(a.matches_played, 0), 2
    ) AS goals_per_match

FROM players p

JOIN appearances a
    ON p.player_id = a.player_id

JOIN teams t
    ON p.team_id = t.team_id

LEFT JOIN goals g
    ON p.player_id = g.player_id

WHERE a.matches_played >= 3

ORDER BY goals_per_match DESC, goals_scored DESC
LIMIT 20;
"""

efficiency_df = pd.read_sql_query(query, conn)

display(efficiency_df)


,player,team,matches_played,goals_scored,goals_per_match
0,Erling HAALAND,Norway,5,7,1.40
1,Kylian MBAPPE,France,8,10,1.25
2,Lionel MESSI,Argentina,8,8,1.00
3,Ismaila SARR,Senegal,4,4,1.00
4,Elijah JUST,New Zealand,3,3,1.00
5,Jude BELLINGHAM,England,8,7,0.88
6,Harry KANE,England,7,6,0.86
7,VINICIUS JUNIOR,Brazil,5,4,0.80
8,Julian QUINONES,Mexico,5,4,0.80
9,Yoane WISSA,Congo DR,4,3,0.75



## 5.6 | Analyzing Goals by Player Position

I compare scoring contributions across player positions to understand how attacking output is distributed between different roles on the field.


In [6]:

# Compare goal contributions by player position
query = """
WITH player_positions AS (
    SELECT
        player_id,
        position,
        COUNT(*) AS appearances,

        ROW_NUMBER() OVER (
            PARTITION BY player_id
            ORDER BY COUNT(*) DESC, position
        ) AS position_rank

    FROM match_appearances
    WHERE appeared = 1
      AND position IS NOT NULL
    GROUP BY player_id, position
),

player_goals AS (
    SELECT
        player_id,
        COUNT(*) AS goals_scored
    FROM player_events
    WHERE event_type = 'goal'
    GROUP BY player_id
)

SELECT
    pp.position,
    COUNT(*) AS total_players,
    SUM(pp.appearances) AS position_appearances,
    SUM(COALESCE(pg.goals_scored, 0)) AS total_goals,

    ROUND(
        1.0 * SUM(COALESCE(pg.goals_scored, 0))
        / NULLIF(SUM(pp.appearances), 0), 2
    ) AS goals_per_appearance

FROM player_positions pp

LEFT JOIN player_goals pg
    ON pp.player_id = pg.player_id

WHERE pp.position_rank = 1

GROUP BY pp.position
ORDER BY total_goals DESC;
"""

position_df = pd.read_sql_query(query, conn)

display(position_df)


,position,total_players,position_appearances,total_goals,goals_per_appearance
0,FW,299,904,170,0.19
1,MF,334,1081,95,0.09
2,DF,360,1088,29,0.03
3,GK,63,215,0,0.00



## 5.7 | Exploring Player Playing Time

I review substitution and participation records to determine whether I can calculate playing time and evaluate player performance on a per-90-minute basis.


In [7]:

# Inspect substitution and playing-time information
query = """
SELECT
    player_name,
    team,
    position,
    is_starter,
    appeared,
    status,
    subbed_on_minutes,
    subbed_off_minutes,
    match_id

FROM match_appearances

WHERE team IN ('Spain', 'Argentina')
  AND (
      subbed_on_minutes IS NOT NULL
      OR subbed_off_minutes IS NOT NULL
  )

LIMIT 20;
"""

minutes_df = pd.read_sql_query(query, conn)

display(minutes_df)


,player_name,team,position,is_starter,appeared,status,subbed_on_minutes,subbed_off_minutes,match_id
0,Ferran TORRES,Spain,FW,1,1,starting,NaN,81.0,2026-M014-ESP-CPV
1,Fabian RUIZ,Spain,MF,1,1,starting,NaN,71.0,2026-M014-ESP-CPV
2,GAVI,Spain,MF,1,1,starting,NaN,71.0,2026-M014-ESP-CPV
3,RODRI,Spain,MF,1,1,starting,NaN,87.0,2026-M014-ESP-CPV
4,Mikel MERINO,Spain,MF,0,1,substitute,71.0,NaN,2026-M014-ESP-CPV
5,Dani OLMO,Spain,FW,0,1,substitute,81.0,NaN,2026-M014-ESP-CPV
6,Nico WILLIAMS,Spain,FW,0,1,substitute,87.0,NaN,2026-M014-ESP-CPV
7,Lamine YAMAL,Spain,FW,0,1,substitute,71.0,NaN,2026-M014-ESP-CPV
8,Gonzalo MONTIEL,Argentina,DF,1,1,starting,NaN,45.0,2026-M019-ARG-ALG
9,Lionel MESSI,Argentina,FW,1,1,starting,NaN,80.0,2026-M019-ARG-ALG



## 5.8 | Validating Playing-Time Records

I check substitution records and match participation to identify missing values and determine whether playing time can be estimated consistently.


In [8]:

# Validate substitution and playing-time records
query = """
SELECT
    COUNT(*) AS total_appearances,

    SUM(CASE
        WHEN is_starter = 1 THEN 1 ELSE 0
    END) AS starters,

    SUM(CASE
        WHEN is_starter = 0 THEN 1 ELSE 0
    END) AS substitutes,

    SUM(CASE
        WHEN is_starter = 1
         AND subbed_off_minutes IS NULL
        THEN 1 ELSE 0
    END) AS full_match_starters,

    SUM(CASE
        WHEN is_starter = 0
         AND subbed_on_minutes IS NULL
        THEN 1 ELSE 0
    END) AS substitutes_missing_entry,

    SUM(CASE
        WHEN subbed_on_minutes > 90
          OR subbed_off_minutes > 90
        THEN 1 ELSE 0
    END) AS extra_time_substitutions

FROM match_appearances
WHERE appeared = 1;
"""

minutes_validation_df = pd.read_sql_query(query, conn)

display(minutes_validation_df)


,total_appearances,starters,substitutes,full_match_starters,substitutes_missing_entry,extra_time_substitutions
0,3288,2288,1000,1294,0,174


### 5.9 | Examining Match Duration

I examine substitution timestamps to identify matches that extended beyond regulation time and assess how this affects player-minute calculations.

In [9]:
# Examine substitution timestamps by match
query = """
SELECT
    m.match_id,
    m.home_team,
    m.away_team,
    m."group" AS stage,
    MAX(ma.subbed_on_minutes) AS latest_sub_on,
    MAX(ma.subbed_off_minutes) AS latest_sub_off,
    MAX(
        COALESCE(ma.subbed_on_minutes, 0),
        COALESCE(ma.subbed_off_minutes, 0)
    ) AS latest_substitution
FROM matches m

JOIN match_appearances ma
    ON m.match_id = ma.match_id

WHERE ma.appeared = 1

GROUP BY
    m.match_id,
    m.home_team,
    m.away_team,
    m."group"

HAVING latest_substitution > 90

ORDER BY latest_substitution DESC;
"""

duration_df = pd.read_sql_query(query, conn)

display(duration_df)

print("Matches with substitutions after 90 minutes:", len(duration_df))

,match_id,home_team,away_team,stage,latest_sub_on,latest_sub_off,latest_substitution
0,2026-M074-GER-PAR,Germany,Paraguay,Round of 32,122.0,122.0,122.0
1,2026-M088-AUS-EGY,Australia,Egypt,Round of 32,121.0,121.0,121.0
2,2026-M096-SUI-COL,Switzerland,Colombia,Round of 16,119.0,119.0,119.0
3,2026-M100-ARG-SUI,Argentina,Switzerland,Quarter-ﬁnal,115.0,115.0,115.0
4,2026-M075-NED-MAR,Netherlands,Morocco,Round of 32,113.0,113.0,113.0
5,2026-M099-NOR-ENG,Norway,England,Quarter-ﬁnal,111.0,111.0,111.0
6,2026-M082-BEL-SEN,Belgium,Senegal,Round of 32,109.0,109.0,109.0
7,2026-M086-ARG-CPV,Argentina,Cabo Verde,Round of 32,104.0,104.0,104.0
8,2026-M104-ESP-ARG,Spain,Argentina,Final,102.0,102.0,102.0
9,2026-M040-NZL-EGY,New Zealand,Egypt,Group G,99.0,99.0,99.0


Matches with substitutions after 90 minutes: 42


### 5.10 | Estimating Player Minutes

I estimate playing time using starting status, substitution timestamps, and match-level duration assumptions. These estimates support player performance comparisons while accounting for potential extra time.

In [10]:
# Estimate match duration and player minutes
query = """
WITH match_duration AS (
    SELECT
        m.match_id,
        CASE
            WHEN m."group" IN (
                'Round of 32',
                'Round of 16',
                'Quarter-final',
                'Semi-final',
                'Final',
                'Bronze final'
            )
            AND MAX(
                COALESCE(ma.subbed_on_minutes, 0),
                COALESCE(ma.subbed_off_minutes, 0)
            ) > 105
            THEN 120
            ELSE 90
        END AS estimated_duration
    FROM matches m
    JOIN match_appearances ma
        ON m.match_id = ma.match_id
    GROUP BY m.match_id
),

player_minutes AS (
    SELECT
        ma.player_id,
        ma.player_name,
        ma.team,
        ma.match_id,
        ma.is_starter,
        md.estimated_duration,

        CASE
            WHEN ma.is_starter = 1
            THEN COALESCE(
                ma.subbed_off_minutes,
                md.estimated_duration
            )

            ELSE md.estimated_duration
                 - ma.subbed_on_minutes
        END AS estimated_minutes

    FROM match_appearances ma

    JOIN match_duration md
        ON ma.match_id = md.match_id

    WHERE ma.appeared = 1
)

SELECT
    player_name,
    team,
    COUNT(*) AS matches_played,
    ROUND(SUM(estimated_minutes), 1) AS total_minutes,
    ROUND(AVG(estimated_minutes), 1) AS avg_minutes_per_match

FROM player_minutes

GROUP BY player_id, player_name, team

HAVING SUM(estimated_minutes) > 0

ORDER BY total_minutes DESC

LIMIT 20;
"""

minutes_df = pd.read_sql_query(query, conn)

display(minutes_df)

,player_name,team,matches_played,total_minutes,avg_minutes_per_match
0,Aymeric LAPORTE,Spain,8,732.0,91.5
1,RODRI,Spain,8,726.0,90.8
2,Emiliano MARTINEZ,Argentina,8,720.0,90.0
3,Marc CUCURELLA,Spain,8,720.0,90.0
4,Pau CUBARSI,Spain,8,720.0,90.0
5,Unai SIMON,Spain,8,720.0,90.0
6,Mike MAIGNAN,France,8,720.0,90.0
7,Kylian MBAPPE,France,8,700.0,87.5
8,Dayot UPAMECANO,France,8,661.0,82.6
9,Alexis MAC ALLISTER,Argentina,8,659.0,82.4


### 5.11 | Validating Estimated Player Minutes

I check estimated playing times for negative values and inconsistencies with assumed match durations before calculating player performance rates.

In [11]:
# Validate estimated playing minutes
query = """
WITH match_duration AS (
    SELECT
        m.match_id,
        CASE
            WHEN m."group" IN (
                'Round of 32',
                'Round of 16',
                'Quarter-final',
                'Semi-final',
                'Final',
                'Bronze final'
            )
            AND MAX(
                COALESCE(ma.subbed_on_minutes, 0),
                COALESCE(ma.subbed_off_minutes, 0)
            ) > 105
            THEN 120
            ELSE 90
        END AS estimated_duration
    FROM matches m
    JOIN match_appearances ma
        ON m.match_id = ma.match_id
    GROUP BY m.match_id
),

player_minutes AS (
    SELECT
        ma.appearance_id,
        ma.match_id,
        ma.player_name,
        ma.team,
        md.estimated_duration,
        CASE
            WHEN ma.is_starter = 1
            THEN COALESCE(
                ma.subbed_off_minutes,
                md.estimated_duration
            )
            ELSE md.estimated_duration - ma.subbed_on_minutes
        END AS estimated_minutes
    FROM match_appearances ma
    JOIN match_duration md
        ON ma.match_id = md.match_id
    WHERE ma.appeared = 1
)

SELECT
    COUNT(*) AS total_appearances,
    SUM(CASE WHEN estimated_minutes < 0 THEN 1 ELSE 0 END)
        AS negative_minutes,
    SUM(CASE WHEN estimated_minutes > estimated_duration THEN 1 ELSE 0 END)
        AS minutes_above_duration,
    SUM(CASE WHEN estimated_minutes = 0 THEN 1 ELSE 0 END)
        AS zero_minutes,
    MIN(estimated_minutes) AS minimum_minutes,
    MAX(estimated_minutes) AS maximum_minutes
FROM player_minutes;
"""

minutes_check_df = pd.read_sql_query(query, conn)

display(minutes_check_df)

,total_appearances,negative_minutes,minutes_above_duration,zero_minutes,minimum_minutes,maximum_minutes
0,3288,87,85,22,-32.0,122.0


### 5.12 | Refining Player-Minute Estimates

I refine estimated playing time by accounting for substitutions, regulation time, and potential extra time. I also check the resulting calculations to ensure that no appearance has negative minutes or exceeds the assumed match duration.

In [12]:
# Refine estimated player minutes
query = """
WITH match_duration AS (
    SELECT
        m.match_id,
        CASE
            WHEN m."group" IN (
                'Round of 32',
                'Round of 16',
                'Quarter-final',
                'Semi-final',
                'Final',
                'Bronze final'
            )
            AND MAX(
                COALESCE(ma.subbed_on_minutes, 0),
                COALESCE(ma.subbed_off_minutes, 0)
            ) > 105
            THEN 120
            ELSE 90
        END AS estimated_duration

    FROM matches m

    JOIN match_appearances ma
        ON m.match_id = ma.match_id

    GROUP BY m.match_id
),

player_minutes AS (
    SELECT
        ma.player_id,
        ma.player_name,
        ma.team,
        ma.match_id,
        md.estimated_duration,

        CASE
            WHEN ma.is_starter = 1
            THEN MIN(
                COALESCE(
                    ma.subbed_off_minutes,
                    md.estimated_duration
                ),
                md.estimated_duration
            )

            ELSE MAX(
                0,
                md.estimated_duration - ma.subbed_on_minutes
            )
        END AS estimated_minutes

    FROM match_appearances ma

    JOIN match_duration md
        ON ma.match_id = md.match_id

    WHERE ma.appeared = 1
)

SELECT
    COUNT(*) AS total_appearances,

    SUM(CASE
        WHEN estimated_minutes < 0 THEN 1
        ELSE 0
    END) AS negative_minutes,

    SUM(CASE
        WHEN estimated_minutes > estimated_duration THEN 1
        ELSE 0
    END) AS minutes_above_duration,

    SUM(CASE
        WHEN estimated_minutes = 0 THEN 1
        ELSE 0
    END) AS zero_minutes,

    ROUND(SUM(estimated_minutes), 1) AS total_player_minutes

FROM player_minutes;
"""

refined_minutes_df = pd.read_sql_query(query, conn)

display(refined_minutes_df)

,total_appearances,negative_minutes,minutes_above_duration,zero_minutes,total_player_minutes
0,3288,0,0,109,205964.0


### 5.13 | Calculating Goals per 90 Minutes

I compare players' scoring efficiency by combining goals scored with estimated playing time. Goals per 90 minutes provides a standardized measure of scoring output across players with different levels of participation.

In [13]:
# Calculate player scoring efficiency
query = """
WITH match_duration AS (
    SELECT
        m.match_id,
        CASE
            WHEN m."group" IN (
                'Round of 32',
                'Round of 16',
                'Quarter-final',
                'Semi-final',
                'Final',
                'Bronze final'
            )
            AND MAX(
                COALESCE(ma.subbed_on_minutes, 0),
                COALESCE(ma.subbed_off_minutes, 0)
            ) > 105
            THEN 120
            ELSE 90
        END AS estimated_duration
    FROM matches m
    JOIN match_appearances ma
        ON m.match_id = ma.match_id
    GROUP BY m.match_id
),

player_minutes AS (
    SELECT
        ma.player_id,
        ma.player_name,
        ma.team,
        CASE
            WHEN ma.is_starter = 1
            THEN MIN(
                COALESCE(ma.subbed_off_minutes, md.estimated_duration),
                md.estimated_duration
            )
            ELSE MAX(
                0,
                md.estimated_duration - ma.subbed_on_minutes
            )
        END AS estimated_minutes
    FROM match_appearances ma
    JOIN match_duration md
        ON ma.match_id = md.match_id
    WHERE ma.appeared = 1
),

minutes_summary AS (
    SELECT
        player_id,
        player_name,
        team,
        COUNT(*) AS matches_played,
        SUM(estimated_minutes) AS total_minutes
    FROM player_minutes
    GROUP BY player_id, player_name, team
),

goals_summary AS (
    SELECT
        player_id,
        COUNT(*) AS goals_scored
    FROM player_events
    WHERE event_type = 'goal'
    GROUP BY player_id
)

SELECT
    ms.player_name AS player,
    ms.team,
    ms.matches_played,
    ROUND(ms.total_minutes, 1) AS estimated_minutes,
    COALESCE(gs.goals_scored, 0) AS goals_scored,
    ROUND(
        90.0 * COALESCE(gs.goals_scored, 0)
        / NULLIF(ms.total_minutes, 0),
        2
    ) AS goals_per_90

FROM minutes_summary ms

LEFT JOIN goals_summary gs
    ON ms.player_id = gs.player_id

WHERE ms.total_minutes >= 180

ORDER BY goals_per_90 DESC

LIMIT 20;
"""

goals_per90_df = pd.read_sql_query(query, conn)

display(goals_per90_df)

,player,team,matches_played,estimated_minutes,goals_scored,goals_per_90
0,Erling HAALAND,Norway,5,450.0,7,1.40
1,Johan MANZAMBI,Switzerland,4,199.0,3,1.36
2,Romelu LUKAKU,Belgium,6,200.0,3,1.35
3,Kylian MBAPPE,France,8,699.0,10,1.29
4,Brian BROBBEY,Netherlands,4,226.0,3,1.19
5,Lionel MESSI,Argentina,8,650.0,8,1.11
6,Ismaila SARR,Senegal,4,337.0,4,1.07
7,Jude BELLINGHAM,England,8,596.0,7,1.06
8,Elijah JUST,New Zealand,3,265.0,3,1.02
9,Amad DIALLO,Côte d'Ivoire,4,184.0,2,0.98


### 5.14 | Comparing Spain and Argentina's Player Efficiency

I compare scoring efficiency among Spain's and Argentina's players to understand how individual contributions differ between the two teams. I use estimated goals per 90 minutes alongside total goals and playing time to identify key attacking contributors.

In [14]:
# Compare scoring efficiency for Spain and Argentina
query_comparison = query.replace(
    "WHERE ms.total_minutes >= 180",
    """WHERE ms.total_minutes >= 180
       AND ms.team IN ('Spain', 'Argentina')"""
).replace(
    "LIMIT 20;",
    "LIMIT 30;"
)

player_comparison_df = pd.read_sql_query(
    query_comparison,
    conn
)

display(player_comparison_df)

,player,team,matches_played,estimated_minutes,goals_scored,goals_per_90
0,Lionel MESSI,Argentina,8,650.0,8,1.11
1,Lautaro MARTINEZ,Argentina,7,282.0,3,0.96
2,Mikel OYARZABAL,Spain,8,606.0,5,0.74
3,Ferran TORRES,Spain,8,253.0,1,0.36
4,Pedro PORRO,Spain,6,534.0,2,0.34
5,Enzo FERNANDEZ,Argentina,7,630.0,2,0.29
6,Fabian RUIZ,Spain,8,321.0,1,0.28
7,Alex BAENA,Spain,7,488.0,1,0.18
8,Julian ALVAREZ,Argentina,8,529.0,1,0.17
9,Cristian ROMERO,Argentina,7,568.0,1,0.16


### 5.15 | Comparing Starting Players and Substitutes

I compare goals scored by starting players and substitutes for Spain and Argentina to understand how each team's attacking contributions are distributed across the squad.

This analysis uses SQL joins and conditional aggregation to connect player appearances with goal events and identify whether goals were scored by starters or substitutes.

In [15]:
# Compare goals scored by starters and substitutes
query = """
SELECT
    ma.team,
    CASE
        WHEN ma.is_starter = 1 THEN 'Starter'
        ELSE 'Substitute'
    END AS player_role,
    COUNT(DISTINCT ma.appearance_id) AS appearances,
    COUNT(pe.event_id) AS goals_scored

FROM match_appearances ma

LEFT JOIN player_events pe
    ON ma.appearance_id = pe.appearance_id
    AND pe.event_type = 'goal'

WHERE ma.team IN ('Spain', 'Argentina')
    AND ma.appeared = 1

GROUP BY
    ma.team,
    player_role

ORDER BY
    ma.team,
    goals_scored DESC;
"""

starter_sub_df = pd.read_sql_query(query, conn)

display(starter_sub_df)

,team,player_role,appearances,goals_scored
0,Argentina,Starter,88,15
1,Argentina,Substitute,42,3
2,Spain,Starter,88,10
3,Spain,Substitute,38,3


### 5.16 | Investigating Goal Discrepancies

I compare match-level goal totals with recorded player goal events to identify discrepancies between team results and individual scoring records.

I also examine own-goal events to determine whether they explain any differences.

In [16]:
# Investigate goal event types for Spain and Argentina
query = """
SELECT
    ma.team,
    pe.event_type,
    COUNT(*) AS total_events

FROM player_events pe

JOIN match_appearances ma
    ON pe.appearance_id = ma.appearance_id

WHERE ma.team IN ('Spain', 'Argentina')
    AND pe.event_type LIKE '%goal%'

GROUP BY
    ma.team,
    pe.event_type

ORDER BY
    ma.team,
    total_events DESC;
"""

goal_events_check_df = pd.read_sql_query(query, conn)

display(goal_events_check_df)

,team,event_type,total_events
0,Argentina,goal,18
1,Spain,goal,13


### 5.17 | Checking Own Goals

I investigate own-goal records to determine whether they explain the differences between match-level scores and player goal-event totals.

In [17]:
# Check own-goal records for Spain and Argentina
query = """
SELECT
    ma.team,
    ma.player_name,
    ma.match_id,
    ma.own_goal_minutes

FROM match_appearances ma

WHERE ma.own_goal_minutes IS NOT NULL
    AND TRIM(CAST(ma.own_goal_minutes AS TEXT)) NOT IN ('', 'None')

ORDER BY
    ma.match_id,
    ma.team;
"""

own_goals_df = pd.read_sql_query(query, conn)

display(own_goals_df)

,team,player_name,match_id,own_goal_minutes
0,Paraguay,Damian BOBADILLA,2026-M004-USA-PAR,7.0
1,Switzerland,Miro MUHEIM,2026-M008-QAT-SUI,95.0
2,Egypt,MOHAMED HANY,2026-M016-BEL-EGY,66.0
3,Iraq,AYMEN HUSSEIN,2026-M018-IRQ-NOR,97.0
4,Jordan,YAZAN ALARAB,2026-M020-AUT-JOR,76.0
5,Qatar,MOHAMED MANAI,2026-M027-CAN-QAT,75.0
6,Australia,Cameron BURGESS,2026-M032-USA-AUS,11.0
7,Saudi Arabia,HASSAN ALTAMBAKTI,2026-M038-ESP-KSA,49.0
8,Uzbekistan,Abduvohid NEMATOV,2026-M047-POR-GHA,60.0
9,Morocco,Yassine BOUNOU,2026-M050-MAR-HAI,10.0


### 5.18 | Reconciling Team Goals and Own Goals

I reconcile player goal events and opponent own goals against the match-level scoring totals for Spain and Argentina. This helps confirm the completeness of the scoring data and prevents own goals from being incorrectly attributed to individual players.

In [18]:
# Reconcile player goals, opponent own goals, and match scores
query = """
WITH match_goals AS (
    SELECT
        home_team AS team,
        SUM(home_score) AS match_goals
    FROM matches
    WHERE home_team IN ('Spain', 'Argentina')
    GROUP BY home_team

    UNION ALL

    SELECT
        away_team AS team,
        SUM(away_score) AS match_goals
    FROM matches
    WHERE away_team IN ('Spain', 'Argentina')
    GROUP BY away_team
),

player_goals AS (
    SELECT
        ma.team,
        COUNT(*) AS player_goals
    FROM player_events pe
    JOIN match_appearances ma
        ON pe.appearance_id = ma.appearance_id
    WHERE pe.event_type = 'goal'
      AND ma.team IN ('Spain', 'Argentina')
    GROUP BY ma.team
),

own_goals AS (
    SELECT
        CASE
            WHEN ma.team = m.home_team THEN m.away_team
            ELSE m.home_team
        END AS benefiting_team,
        COUNT(*) AS own_goals
    FROM match_appearances ma
    JOIN matches m
        ON ma.match_id = m.match_id
    WHERE ma.own_goal_minutes IS NOT NULL
      AND TRIM(CAST(ma.own_goal_minutes AS TEXT))
          NOT IN ('', 'None')
    GROUP BY benefiting_team
)

SELECT
    mg.team,
    SUM(mg.match_goals) AS match_goals,
    COALESCE(pg.player_goals, 0) AS player_goals,
    COALESCE(og.own_goals, 0) AS opponent_own_goals,
    COALESCE(pg.player_goals, 0)
        + COALESCE(og.own_goals, 0) AS reconciled_goals,
    SUM(mg.match_goals)
        - COALESCE(pg.player_goals, 0)
        - COALESCE(og.own_goals, 0) AS discrepancy

FROM match_goals mg

LEFT JOIN player_goals pg
    ON mg.team = pg.team

LEFT JOIN own_goals og
    ON mg.team = og.benefiting_team

GROUP BY mg.team
ORDER BY mg.team;
"""

goals_reconciliation_df = pd.read_sql_query(query, conn)

display(goals_reconciliation_df)

,team,match_goals,player_goals,opponent_own_goals,reconciled_goals,discrepancy
0,Argentina,19,18,1,19,0
1,Spain,14,13,1,14,0


### 5.19 | Comparing Goal Contributions by Player

I analyze how individual players contributed to Spain's and Argentina's total goals, identifying each team's leading scorers and measuring their share of overall team scoring.

This analysis excludes own goals from individual player totals while retaining them in the team's total match goals.



In [19]:
# Calculate individual contributions to team scoring
query = """
WITH player_goals AS (
    SELECT
        ma.team,
        ma.player_name AS player,
        COUNT(*) AS goals_scored
    FROM player_events pe
    JOIN match_appearances ma
        ON pe.appearance_id = ma.appearance_id
    WHERE pe.event_type = 'goal'
      AND ma.team IN ('Spain', 'Argentina')
    GROUP BY ma.team, ma.player_id, ma.player_name
),

team_goals AS (
    SELECT
        team,
        SUM(goals) AS total_goals
    FROM (
        SELECT home_team AS team, home_score AS goals
        FROM matches

        UNION ALL

        SELECT away_team AS team, away_score AS goals
        FROM matches
    )
    WHERE team IN ('Spain', 'Argentina')
    GROUP BY team
)

SELECT
    pg.team,
    pg.player,
    pg.goals_scored,
    tg.total_goals AS team_goals,
    ROUND(
        100.0 * pg.goals_scored / NULLIF(tg.total_goals, 0),
        1
    ) AS contribution_pct

FROM player_goals pg
JOIN team_goals tg
    ON pg.team = tg.team

ORDER BY pg.team, pg.goals_scored DESC;
"""

contributions_df = pd.read_sql_query(query, conn)

display(contributions_df)

,team,player,goals_scored,team_goals,contribution_pct
0,Argentina,Lionel MESSI,8,19,42.1
1,Argentina,Lautaro MARTINEZ,3,19,15.8
2,Argentina,Enzo FERNANDEZ,2,19,10.5
3,Argentina,Alexis MAC ALLISTER,1,19,5.3
4,Argentina,Cristian ROMERO,1,19,5.3
5,Argentina,Giovani LO CELSO,1,19,5.3
6,Argentina,Julian ALVAREZ,1,19,5.3
7,Argentina,Lisandro MARTINEZ,1,19,5.3
8,Spain,Mikel OYARZABAL,5,14,35.7
9,Spain,Mikel MERINO,2,14,14.3
